# EM Displacement VLM — Colab A100

**Required runtime:** Runtime → Change runtime type → GPU → **A100**.

This notebook is the primary compute entrypoint: Drive persistence, clone, Unsloth, data freeze, Gemma 3-4B LoRA FT (`r=32`), sanity check, Hub push.

Docs: [`docs/COLAB_A100.md`](https://github.com/rlogger/em-displacement-vlm/blob/main/docs/COLAB_A100.md)

## 0. Assert A100

In [ ]:
!nvidia-smi

import torch
assert torch.cuda.is_available(), "No CUDA GPU — enable a GPU runtime."
name = torch.cuda.get_device_name(0)
print("GPU:", name)
print("bf16 supported:", torch.cuda.is_bf16_supported())
if "A100" not in name:
    raise SystemExit(
        f"Refusing to continue on '{name}'. Switch runtime to A100 before FT."
    )
print("A100 OK")

## 1. Mount Drive (wipe insurance)

In [ ]:
from pathlib import Path
import os

MOUNT_DRIVE = True
DRIVE_PROJECT = Path("/content/drive/MyDrive/em-displacement-vlm")

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    for sub in ("data", "checkpoints", "results", "activations", "judge_cache"):
        (DRIVE_PROJECT / sub).mkdir(parents=True, exist_ok=True)
    os.environ["EM_DATA_DIR"] = str(DRIVE_PROJECT / "data")
    os.environ["EM_CHECKPOINT_DIR"] = str(DRIVE_PROJECT / "checkpoints")
    os.environ["EM_RESULTS_DIR"] = str(DRIVE_PROJECT / "results")
    print("Drive project:", DRIVE_PROJECT)
else:
    print("WARNING: Drive not mounted — session wipe will delete checkpoints.")

## 2. Clone / pull repo

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/rlogger/em-displacement-vlm.git"
REPO_DIR = Path("/content/em-displacement-vlm")
BRANCH = "main"

if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    %cd {REPO_DIR}
    !git fetch origin
    !git checkout {BRANCH}
    !git pull --ff-only origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

!git rev-parse --short HEAD
!git status -sb

## 3. Install Unsloth + project

Colab already ships PyTorch + CUDA. Install Unsloth for the current torch build, then the editable package.

In [ ]:
import torch, re, subprocess, sys

v = re.match(r"[\d]+\.[\d]+", torch.__version__).group(0)
print("torch", torch.__version__, "→ unsloth wheel tag", v)

# Official Unsloth Colab pattern (adjust if Unsloth docs change).
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "unsloth",
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "-e", ".[vlm,dev]",
])

from em_displacement_vlm.runtime import runtime_info
from em_displacement_vlm.paths import data_dir, checkpoint_dir, results_dir

for k, v in runtime_info().items():
    print(f"{k}: {v}")
print("data_dir:", data_dir())
print("checkpoint_dir:", checkpoint_dir())
print("results_dir:", results_dir())

## 4. Secrets

In [ ]:
from google.colab import userdata
import os

def _set_secret(name: str, required: bool = False) -> None:
    try:
        os.environ[name] = userdata.get(name)
        print(f"Loaded secret: {name}")
    except Exception:
        msg = f"Secret not set: {name}"
        if required:
            raise SystemExit(msg + " (required for A100 FT / Hub push)")
        print(msg + " (ok if unused)")

_set_secret("HF_TOKEN", required=True)
_set_secret("WANDB_API_KEY", required=False)
_set_secret("GITHUB_TOKEN", required=False)

!huggingface-cli login --token "$HF_TOKEN" --add-to-git-credential

## 5. Freeze data (hash-disjoint roles)

Writes `utk_harmful.jsonl`, `neutral_faces.jsonl`, and Role 1–3 splits under `EM_DATA_DIR`.

In [ ]:
!python scripts/prepare_datasets.py --use-hf
!python scripts/check_disjointness.py

## 6. Configure Hub repo id for this run

Edit `HUB_REPO` before fine-tuning so adapters land on your account (wipe insurance).

In [ ]:
from pathlib import Path
import yaml

HUB_REPO = "YOUR_HF_USER/FT_R32_gemma3_faces_colab"  # ← change me
SEED = 42  # also run 43, 44 for the n=3 matrix

cfg_path = Path("configs/colab_a100.yaml")
cfg = yaml.safe_load(cfg_path.read_text())
cfg["hub_repo"] = HUB_REPO
cfg["seed"] = SEED
cfg["run_name"] = f"colab_a100_ft_r32_seed{SEED}"
cfg["push_to_hub"] = True
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(cfg_path.read_text())

## 7. Fine-tune Gemma 3-4B → `M_ft` (r=32)

Uses `configs/colab_a100.yaml`. Expect ~1 epoch on 1,500 faces. Push happens at the end of the script when `push_to_hub: true`.

In [ ]:
!python scripts/ft_faces.py --config configs/colab_a100.yaml

## 8. Sanity-check EM (held-out prompts only)

In [ ]:
# Prefer the Hub id you just pushed; fallback to local FT_R32 checkpoint.
MODEL_ID = HUB_REPO  # from cell above

!python scripts/sanity_check_em.py --config configs/sanity_em.yaml --model-id {MODEL_ID}

## 9. Next on this A100 session

1. Repeat FT for seeds **43** and **44** (edit `SEED` above).
2. RQ1 extraction — wire `configs/extract_rq1.yaml` to your `M_ft` adapter.
3. BLOCK-EM — `configs/block_em.yaml` (λ ∈ {0.1, 1, 10}).
4. Keep Drive mounted; re-pull git after local doc/code edits.

Do **not** leave adapters only on `/content` — Hub + Drive only survive wipe.